In [33]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from bs4 import BeautifulSoup
import json
import time
import csv
import re
import os
from datetime import datetime
from typing import Dict, List, Optional
import random
from pathlib import Path

class YandexRealtyScraper:
    def __init__(self, headless=True, download_images=False):
        self.download_images = download_images
        self.setup_driver(headless)
        self.base_url = "https://realty.yandex.ru"
        
    def setup_driver(self, headless=True):
        chrome_options = Options()
        if headless:
            chrome_options.add_argument("--headless")
        chrome_options.add_argument("--no-sandbox")
        chrome_options.add_argument("--disable-dev-shm-usage")
        chrome_options.add_argument("--disable-blink-features=AutomationControlled")
        chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])
        chrome_options.add_experimental_option('useAutomationExtension', False)
        chrome_options.add_argument("--disable-gpu")
        chrome_options.add_argument("--window-size=1920,1080")
        chrome_options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")
        
        if self.download_images:
            download_dir = os.path.join(os.getcwd(), "downloaded_images")
            os.makedirs(download_dir, exist_ok=True)
            prefs = {
                "download.default_directory": download_dir,
                "download.prompt_for_download": False,
                "download.directory_upgrade": True,
                "safebrowsing.enabled": True
            }
            chrome_options.add_experimental_option("prefs", prefs)
        
        self.driver = webdriver.Chrome(options=chrome_options)
        self.wait = WebDriverWait(self.driver, 10)
        
    def __del__(self):
        if hasattr(self, 'driver'):
            self.driver.quit()
    
    def get_listings(self, city: str = "moskva", pages: int = 5, max_offers: int = 100) -> List[str]:
        listing_urls = []
        
        for page in range(1, pages + 1):
            if len(listing_urls) >= max_offers:
                break
                
            try:
                if page == 1:
                    url = f"{self.base_url}/{city}/kupit/kvartira/"
                else:
                    url = f"{self.base_url}/{city}/kupit/kvartira/?page={page}"
                
                print(f" Загружаем страницу {page}: {url}")
                self.driver.get(url)
                
                time.sleep(random.uniform(3, 5))
                
                for _ in range(3):
                    self.driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
                    time.sleep(1)
                
                links = self._extract_offer_links()
                
                if links:
                    listing_urls.extend(links)
                    print(f"   Найдено {len(links)} объявлений на странице {page}")
                else:
                    print(f"   Не найдено объявлений на странице {page}")
                    self.driver.save_screenshot(f"debug_page_{page}.png")
                
                time.sleep(random.uniform(2, 4))
                
            except Exception as e:
                print(f" Ошибка при загрузке страницы {page}: {e}")
                continue
        
        listing_urls = list(set(listing_urls))[:max_offers]
        print(f"\n Всего найдено уникальных объявлений: {len(listing_urls)}")
        return listing_urls
    
    def _extract_offer_links(self) -> List[str]:
        """Извлекает ссылки на объявления разными способами"""
        links = []
        
        try:
            cards = self.driver.find_elements(By.CSS_SELECTOR, '[data-testid="offer-card"]')
            for card in cards:
                try:
                    link = card.find_element(By.TAG_NAME, 'a').get_attribute('href')
                    if link and '/offer/' in link:
                        links.append(link)
                except:
                    pass
        except:
            pass
        
        if not links:
            elements = self.driver.find_elements(By.CSS_SELECTOR, "a[href*='/offer/']")
            for elem in elements:
                href = elem.get_attribute('href')
                if href and href not in links:
                    links.append(href)
        
        if not links:
            soup = BeautifulSoup(self.driver.page_source, 'html.parser')
            for a in soup.find_all('a', href=True):
                if '/offer/' in a['href']:
                    full_url = a['href'] if a['href'].startswith('http') else self.base_url + a['href']
                    if full_url not in links:
                        links.append(full_url)
        
        return links
    
    def parse_structured_data(self, soup: BeautifulSoup) -> Dict:
        data = {}
        
        scripts = soup.find_all('script', type='application/ld+json')
        for script in scripts:
            try:
                json_data = json.loads(script.string)
                if isinstance(json_data, dict):
                    if 'offers' in json_data:
                        if isinstance(json_data['offers'], dict):
                            data['price'] = json_data['offers'].get('price')
                        elif isinstance(json_data['offers'], list) and json_data['offers']:
                            data['price'] = json_data['offers'][0].get('price')
                    
                    if 'geo' in json_data:
                        data['latitude'] = json_data['geo'].get('latitude')
                        data['longitude'] = json_data['geo'].get('longitude')
                    
                    if 'description' in json_data:
                        data['json_description'] = json_data['description']
            except:
                continue
        
        price_elem = soup.find('span', {'data-testid': 'price-value'})
        if price_elem:
            price_text = price_elem.get_text(strip=True)
            price_match = re.search(r'[\d\s]+', price_text)
            if price_match:
                data['price'] = int(re.sub(r'[^\d]', '', price_match.group()))
        
        area_elem = soup.find('span', {'data-testid': 'total-area'})
        if area_elem:
            area_text = area_elem.get_text(strip=True)
            area_match = re.search(r'[\d.,]+', area_text)
            if area_match:
                try:
                    data['total_area'] = float(area_match.group().replace(',', '.'))
                except:
                    pass
        
        floor_elem = soup.find('span', {'data-testid': 'floor'})
        if floor_elem:
            floor_text = floor_elem.get_text(strip=True)
            floors = re.findall(r'\d+', floor_text)
            if floors:
                data['floor'] = int(floors[0])
            if len(floors) > 1:
                data['total_floors'] = int(floors[1])
        
        rooms_elem = soup.find('span', {'data-testid': 'rooms'})
        if rooms_elem:
            rooms_text = rooms_elem.get_text(strip=True).lower()
            if 'студия' in rooms_text:
                data['rooms'] = 'студия'
            else:
                rooms_match = re.search(r'\d+', rooms_text)
                if rooms_match:
                    data['rooms'] = int(rooms_match.group())
        
        address_elem = soup.find('span', {'data-testid': 'address'})
        if address_elem:
            data['address'] = address_elem.get_text(strip=True)
        
        return data
    
    def parse_description(self, soup: BeautifulSoup) -> str:
        selectors = [
            ('div', {'data-testid': 'offer-description'}),
            ('div', {'class': 'OfferDescription'}),
            ('div', {'itemprop': 'description'}),
            ('div', {'class': re.compile(r'.*description.*', re.I)})
        ]
        
        for tag, attrs in selectors:
            elem = soup.find(tag, attrs)
            if elem:
                text = elem.get_text(strip=True)
                if text:
                    return text
        
        return ""
    
    def parse_images(self, soup: BeautifulSoup) -> List[str]:
        images = []
        
        scripts = soup.find_all('script', type='application/ld+json')
        for script in scripts:
            try:
                json_data = json.loads(script.string)
                if 'image' in json_data:
                    if isinstance(json_data['image'], list):
                        images.extend(json_data['image'])
                    elif isinstance(json_data['image'], str):
                        images.append(json_data['image'])
            except:
                continue
        
        gallery = soup.find('div', {'data-testid': 'gallery'})
        if gallery:
            for img in gallery.find_all('img'):
                src = img.get('src') or img.get('data-src')
                if src and src.startswith('http') and 'blob:' not in src:
                    src = src.split('?')[0]
                    if src not in images:
                        images.append(src)
        
        if not images:
            for img in soup.find_all('img'):
                src = img.get('src') or img.get('data-src')
                if src and src.startswith('http') and 'avatar' not in src and 'logo' not in src:
                    src = src.split('?')[0]
                    if src not in images:
                        images.append(src)
        
        return images[:20] 
    
    def scrape_listing(self, url: str) -> Optional[Dict]:
        print(f"   Сбор данных: {url.split('/')[-1]}")
        
        try:
            self.driver.get(url)
            time.sleep(random.uniform(2, 4))
            
            self.driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(1)
            
            soup = BeautifulSoup(self.driver.page_source, 'html.parser')
            
            structured = self.parse_structured_data(soup)
            
            data = {
                'url': url,
                'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
                'title': self._get_title(soup),
                'description': self.parse_description(soup),
                'images': self.parse_images(soup),
                **structured
            }
            
            return data
            
        except Exception as e:
            print(f"   Ошибка парсинга: {e}")
            return None
    
    def _get_title(self, soup: BeautifulSoup) -> str:
        title_elem = soup.find('h1')
        if title_elem:
            return title_elem.get_text(strip=True)
        return ""
    
    def save_to_csv(self, data: List[Dict], filename: str = "realty_data.csv"):
        if not data:
            print(" Нет данных для сохранения")
            return
        
        fieldnames = [
            'timestamp', 'title', 'price', 'total_area', 'rooms',
            'floor', 'total_floors', 'address', 'latitude', 'longitude',
            'description', 'json_description', 'url', 'images_count'
        ]
        
        with open(filename, 'w', newline='', encoding='utf-8-sig') as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction='ignore')
            writer.writeheader()
            
            for item in data:
                row = {}
                for field in fieldnames:
                    value = item.get(field, '')
                    if value is None:
                        value = ''
                    if field == 'images_count':
                        row[field] = len(item.get('images', []))
                    else:
                        row[field] = value
                writer.writerow(row)
        
        print(f"\n Данные сохранены в {filename}")
        print(f"   Всего записей: {len(data)}")
    
    def save_images_info(self, data: List[Dict], filename: str = "images_info.json"):
        images_data = []
        for item in data:
            if item.get('images'):
                images_data.append({
                    'url': item['url'],
                    'title': item.get('title', ''),
                    'images_count': len(item['images']),
                    'images': item['images'][:10]
                })
        
        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(images_data, f, ensure_ascii=False, indent=2)
        
        print(f" Информация об изображениях сохранена в {filename}")
        print(f"   Объявлений с изображениями: {len(images_data)}")
    
    def print_statistics(self, data: List[Dict]):
        if not data:
            return
        
        total = len(data)
        with_desc = sum(1 for item in data if item.get('description'))
        with_images = sum(1 for item in data if item.get('images'))
        with_price = sum(1 for item in data if item.get('price'))
        
        print(" СТАТИСТИКА СБОРА ДАННЫХ")
        print(f" Всего объявлений: {total}")
        print(f" С описанием: {with_desc} ({with_desc/total*100:.1f}%)")
        print(f" С изображениями: {with_images} ({with_images/total*100:.1f}%)")
        print(f" С указанием цены: {with_price} ({with_price/total*100:.1f}%)")
        
        prices = [item['price'] for item in data if item.get('price')]
        if prices:
            avg_price = sum(prices) / len(prices)
            print(f"\n Цены:")
            print(f"   Средняя: {avg_price:,.0f} ₽")
            print(f"   Минимальная: {min(prices):,.0f} ₽")
            print(f"   Максимальная: {max(prices):,.0f} ₽")
        
        areas = [item['total_area'] for item in data if item.get('total_area')]
        if areas:
            print(f"\n Площади:")
            print(f"   Средняя: {sum(areas)/len(areas):.1f} м²")
            print(f"   Минимальная: {min(areas):.1f} м²")
            print(f"   Максимальная: {max(areas):.1f} м²")
        

def main():
    print(" Яндекс.Недвижимость - Сбор данных")
    
    CITY = "moskva"
    PAGES = 13
    MAX_OFFERS = 250
    
    scraper = YandexRealtyScraper(headless=False)
    
    try:
        print(f"\n Город: {CITY}")
        print(f" Страниц: {PAGES}")
        print(f" Максимум объявлений: {MAX_OFFERS}\n")
        
        listing_urls = scraper.get_listings(city=CITY, pages=PAGES, max_offers=MAX_OFFERS)
        
        if not listing_urls:
            print(" Не найдено объявлений. Проверьте подключение.")
            return
        
        print(f"\n Начинаем сбор данных с {len(listing_urls)} объявлений...\n")
        scraped_data = []
        
        for i, url in enumerate(listing_urls, 1):
            print(f"\n[{i}/{len(listing_urls)}]", end=" ")
            data = scraper.scrape_listing(url)
            
            if data:
                scraped_data.append(data)
                price = data.get('price', 'N/A')
                if price != 'N/A':
                    price = f"{price:,} ₽"
                area = data.get('total_area', 'N/A')
                print(f"   Цена: {price} | Площадь: {area} м² | Изображений: {len(data.get('images', []))}")
            else:
                print(f"   Не удалось собрать данные")
        
        if scraped_data:
            scraper.save_to_csv(scraped_data)
            scraper.save_images_info(scraped_data)
            scraper.print_statistics(scraped_data)
            
            print("\n Пример собранных данных:")
            sample = scraped_data[0]
            print(f"Заголовок: {sample.get('title', 'N/A')}")
            print(f"Цена: {sample.get('price', 'N/A'):,} ₽" if sample.get('price') else "Цена: не указана")
            print(f"Площадь: {sample.get('total_area', 'N/A')} м²")
            print(f"Комнат: {sample.get('rooms', 'N/A')}")
            print(f"Этаж: {sample.get('floor', 'N/A')} из {sample.get('total_floors', 'N/A')}")
            print(f"Адрес: {sample.get('address', 'N/A')}")
            print(f"Описание: {sample.get('description', '')[:150]}..." if sample.get('description') else "Описание отсутствует")
            print(f"Изображений: {len(sample.get('images', []))}")
            print(f"Ссылка: {sample.get('url', 'N/A')}")
            
        else:
            print(" Не удалось собрать данные")
            
    except KeyboardInterrupt:
        print("\n\n Прервано пользователем")
    except Exception as e:
        print(f"\n Ошибка: {e}")
        import traceback
        traceback.print_exc()
    finally:
        scraper.driver.quit()
        print("\n Скрипт завершен")

if __name__ == "__main__":
    main()

 Яндекс.Недвижимость - Сбор данных

 Город: moskva
 Страниц: 13
 Максимум объявлений: 250

 Загружаем страницу 1: https://realty.yandex.ru/moskva/kupit/kvartira/
   Найдено 23 объявлений на странице 1
 Загружаем страницу 2: https://realty.yandex.ru/moskva/kupit/kvartira/?page=2
   Найдено 23 объявлений на странице 2
 Загружаем страницу 3: https://realty.yandex.ru/moskva/kupit/kvartira/?page=3
   Найдено 23 объявлений на странице 3
 Загружаем страницу 4: https://realty.yandex.ru/moskva/kupit/kvartira/?page=4
   Найдено 22 объявлений на странице 4
 Загружаем страницу 5: https://realty.yandex.ru/moskva/kupit/kvartira/?page=5
   Найдено 23 объявлений на странице 5
 Загружаем страницу 6: https://realty.yandex.ru/moskva/kupit/kvartira/?page=6
   Найдено 23 объявлений на странице 6
 Загружаем страницу 7: https://realty.yandex.ru/moskva/kupit/kvartira/?page=7
   Найдено 23 объявлений на странице 7
 Загружаем страницу 8: https://realty.yandex.ru/moskva/kupit/kvartira/?page=8
   Найдено 23 объяв